# Chapter 5 · Findings and the inspection report


**Learning outcomes:** Combine saved results, distinguish checks from issues, follow evidence references, state a supported conclusion, and export a manageable Excel report.

**Checks in this chapter:** No new checks. This chapter reads CHK01–CHK14 results from Chapters 2–4.


## Block 1 · Confirm the report inputs

**Objective:** Identify what this notebook reads and what it creates.

**Expectation:** All fourteen saved checks and their corresponding issue/evidence files should exist for the same source.

**Where it comes from:** The fixed catalogue and explicit file workflow. Run Chapters 2, 3 and 4 before this notebook.

**Objects used here:** `raw` will be the source table. check_catalogue will be the fourteen-row reference. Saved results provide all inspection counts; checks are not rerun here.

**Method reference:** [read_csv()](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html) · [Path](https://docs.python.org/3/library/pathlib.html#pathlib.Path) · [len()](https://docs.python.org/3/library/functions.html#len)


In [ ]:
import pandas as pd
from pathlib import Path


# Start from the same unchanged teaching file in every notebook.
# Paths are relative to the study-guide folder opened in VS Code.
DATASET_PATH = Path("Data/wildfires.csv")
raw = pd.read_csv(DATASET_PATH, low_memory=False)

# An empty table cannot show that values passed a check.
if len(raw) == 0:
    raise ValueError("No records were loaded. Check Data/wildfires.csv before continuing.")

print("Source file:", DATASET_PATH)
print("Imported records:", len(raw))
print("Columns:", raw.columns.tolist())


In [ ]:
check_catalogue = pd.read_csv("Check_Catalogue.csv")
# Confirm that earlier chapters saved the files needed for this report.
# A named missing file tells us exactly which previous check needs to be run.
for check_id in check_catalogue["Check ID"]:
    for result_folder in ["checks", "issues", "evidence", "previews"]:
        required_file = Path("Outputs") / result_folder / f"{check_id}.csv"
        if not required_file.exists():
            raise FileNotFoundError(
                f"Run Chapters 2–4 first. Required result is missing: {required_file}"
            )
print("All fourteen checks have saved result files.")


**Required packages:** Pandas performs the table work. `openpyxl` is the additional Excel
writer. `ipykernel` supplies the VS Code notebook kernel. The setup command is shown in
the project README and below for reference; do not reinstall at every chapter.

```bash
python -m pip install "pandas==3.0.5" openpyxl ipykernel
```

In the selected notebook kernel, `%pip install "pandas==3.0.5" openpyxl ipykernel` is an
alternative. Restart the kernel after installation, then run the notebook from the top.
`pathlib` is part of Python. NumPy, Matplotlib and statistical packages are not used by the guide.


## Block 2 · Combine saved check and issue tables

**Objective:** Build one complete check log and one issue log from saved chapter results.

**Expectation:** Every check has a check row. Only a finding or limitation needing attention creates an issue row.

**Where it comes from:** The reporting structure and the save_check function taught in Chapter 2.

**Objects used here:** `check_catalogue` lists the input references and was loaded immediately above. check_parts and issue_parts will be lists of small DataFrames, not source records.

**Method reference:** [read_csv()](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html) · [concat()](https://pandas.pydata.org/docs/reference/api/pandas.concat.html) · [DataFrame()](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html) · [head()](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.head.html)


In [ ]:
check_parts = []
issue_parts = []
for check_id in check_catalogue["Check ID"]:
    # Each check CSV contains one summary row. Its issue CSV may have zero rows.
    check_part = pd.read_csv(f"Outputs/checks/{check_id}.csv")
    issue_part = pd.read_csv(f"Outputs/issues/{check_id}.csv")
    check_parts.append(check_part)
    if not issue_part.empty:
        issue_parts.append(issue_part)

# concat stacks tables with the same headings. ignore_index supplies simple new
# log-row labels; evidence keeps its separate original source_row references.
check_log = pd.concat(check_parts, ignore_index=True)
if len(issue_parts) > 0:
    issue_log = pd.concat(issue_parts, ignore_index=True)
else:
    # Keep the agreed issue headings even if this dataset produces no concerns.
    issue_log = pd.read_csv("Outputs/issues/CHK01.csv").head(0)

display(check_log)
display(issue_log)
# Save the assembled tables once. Their source counts came from earlier checks.
check_log.to_csv("Outputs/Check_Log.csv", index=False)
issue_log.to_csv("Outputs/Issue_Log.csv", index=False)


**Read the result:** The check log contains fourteen checks. The issue log can be shorter because fully passed checks do not need invented issues. Passed checks with Untested records still have a limitation to explain, so they can appear in the issue log.



**Pause and explain:** Why is Check_Log row count not a count of fire records?

<details>
<summary>Suggested answer — open after discussing</summary>

Each row means one check. The fire-record counts are in its named count columns.

</details>


## Block 3 · Read coverage as well as flags

**Objective:** Interpret the three outcome counts and the scope of a Passed result.

**Expectation:** Passed, Flagged and Untested partition the stated record population; Flagged is part of Checked.

**Where it comes from:** The outcome definitions from Chapter 1.

**Objects used here:** `check_log` is the assembled fourteen-row table created in Block 2. Each row describes a check, not a fire. The column names below state exactly which record counts are shown.

**Method reference:** [loc](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.loc.html) · [DataFrame()](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html)


In [ ]:
# This reads saved count columns; it does not repeat any inspection condition.
coverage_view = check_log[[
    "Check ID", "Check", "Passed records", "Flagged records", "Untested records", "Check result"
]]
display(coverage_view)

# A visible arithmetic comparison is enough for this introductory reporting task.
# This checks our count accounting, not the truth of the original measurements.
count_review = check_log[["Check ID", "Total records", "Checked records"]].copy()
count_review["Passed + Flagged"] = check_log["Passed records"] + check_log["Flagged records"]
count_review["Checked + Untested"] = check_log["Checked records"] + check_log["Untested records"]
display(count_review)


**Read the result:** Compare Passed + Flagged with Checked, and Checked + Untested with Total. The same record may belong to several checks, so counts from different rows must not be added to claim a distinct problem-record total. Review any different source totals and rerun the earlier chapters using the same dataset.



**Pause and explain:** Why can CHK13 be Passed and still contribute an issue row?

<details>
<summary>Suggested answer — open after discussing</summary>

Its available pairs may all meet date order while some records lack a usable pair. The issue records that limitation.

</details>


## Block 4 · Follow evidence without confusing a preview with a population

**Objective:** Trace a saved concern to original values using a check reference and source-row label.

**Expectation:** The CSV retains complete evidence; the displayed preview contains only a few examples.

**Where it comes from:** The source-preserving evidence model. source_row is a Pandas import label, not a fire identifier or physical line number.

**Objects used here:** The following cell loads saved CHK13 files directly, so it does not depend on a date-order mask from Chapter 4.

**Method reference:** [read_csv()](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html) · [head()](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.head.html)


In [ ]:
date_order_summary = pd.read_csv("Outputs/checks/CHK13.csv")
date_order_evidence_preview = pd.read_csv("Outputs/previews/CHK13.csv")
display(date_order_summary)
display(date_order_evidence_preview)
print("Preview records displayed:", len(date_order_evidence_preview))
print("Full evidence CSV: Outputs/evidence/CHK13.csv")


**Read the result:** The summary measures the complete stated population. The preview illustrates its concerns. A source_row value points back to the unchanged import, even though the preview has its own simple display index.



**Pause and explain:** If the preview shows five unavailable pairs, does that mean only five records were Untested?

<details>
<summary>Suggested answer — open after discussing</summary>

No. The Untested column is the full count. The saved preview contains examples only.

</details>


## Block 5 · Export a manageable Excel report

**Objective:** Write the catalogue, logs and clearly labelled evidence previews into one workbook.

**Expectation:** Excel previews illustrate concerns. Complete record evidence remains in the named CSV files.

**Where it comes from:** Our teaching reporting choice: full evidence can be very large, so it is not split into scores of Excel sheets.

**Objects used here:** `check_catalogue`, `check_log` and `issue_log` are the saved-reference and combined tables from Blocks 1–2. evidence_previews will be a dictionary pairing sheet names with short loaded tables.

**Method reference:** [read_csv()](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html) · [DataFrame()](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html) · [ExcelWriter()](https://pandas.pydata.org/docs/reference/api/pandas.ExcelWriter.html) · [to_excel()](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.to_excel.html)


**What the workbook contains:** `Workbook_Guide`, `Check_Catalogue`, `Check_Log`, `Issue_Log`,
and one `Evidence_CHKnn` preview sheet for each concern. There are at most 18 sheets for these
fourteen checks. Each preview keeps up to five Flagged and five Untested examples, not complete evidence.

**Why separate CSV evidence?** A full missing-date or missing-name table can contain hundreds
of thousands of records. CSV preserves it simply. The Excel report remains small and readable.
Include the full `Outputs/evidence` folder when complete evidence is required for submission.
If your assessment instructions require a different evidence scope, follow those instructions
and describe the scope explicitly.


In [ ]:
# Each issue has the same fixed reference as its check and evidence file.
# Only a small preview is loaded; the full CSV stays available beside the report.
evidence_previews = {}
for check_id in issue_log["Check ID"]:
    preview_file = f"Outputs/previews/{check_id}.csv"
    evidence_previews[f"Evidence_{check_id}"] = pd.read_csv(preview_file)

workbook_guide = pd.DataFrame({
    "Item": ["Source", "Imported records", "Check scope", "Evidence scope", "Complete evidence", "Row reference", "Outcome meaning"],
    "Explanation": [
        "Data/wildfires.csv", str(len(raw)),
        "Fourteen core record checks; one row per check in Check_Log.",
        "Every Evidence_CHKnn sheet is a PREVIEW: up to five Flagged and five Untested records.",
        "Outputs/evidence/CHKnn.csv contains every concern record for that check.",
        "source_row is the unchanged Pandas import label, not a fire ID or physical CSV line.",
        "Passed applies to checked records. Untested counts remain limitations."
    ]
})
report_path = Path("Outputs/Wildfires_Inspection_Report.xlsx")
# One writer keeps all sheets in one workbook. index=False omits extra display
# indices; evidence already has its explicit source_row column from the CSV.
with pd.ExcelWriter(report_path, engine="openpyxl") as writer:
    workbook_guide.to_excel(writer, sheet_name="Workbook_Guide", index=False)
    check_catalogue.to_excel(writer, sheet_name="Check_Catalogue", index=False)
    check_log.to_excel(writer, sheet_name="Check_Log", index=False)
    issue_log.to_excel(writer, sheet_name="Issue_Log", index=False)
    for sheet_name, preview_table in evidence_previews.items():
        preview_table.to_excel(writer, sheet_name=sheet_name, index=False)
print("Saved:", report_path)
print("Evidence sheets are previews; complete evidence remains in Outputs/evidence.")


**Read the result:** One workbook contains the reference, counts, explanations and previews. The writer closes and saves when the with block ends. Close an already open copy in Excel before regenerating it.



**Pause and explain:** Why do we omit the display index while retaining source_row?

<details>
<summary>Suggested answer — open after discussing</summary>

The display index only numbers preview rows. source_row locates the original source record.

</details>


## Block 6 · Write conclusions that the checks support

**Objective:** Explain usability and limitations for each analysis question.

**Conclusion pattern:** “For Q[number], [specific fields] can support [specific comparison]
for [stated population]. We found [named concern or limitation]. Before analysis we need
[justified decision or investigation].”

| Question | What your conclusion should address |
| --- | --- |
| Q1: yearly and state counts | Year readiness, whole/in-period years, year agreement, state codes and repeated records. |
| Q2: size by reported cause | Ordinary positive size values, undefined causes and repeated records. |
| Q3: discovery-to-containment duration | Availability, readability, date order and the number of usable pairs. |
| Q4: display names | Absent names and outside spaces; record identity remains a separate concern. |

**Student activity:** Write one conclusion for Q3 using CHK13 counts, then one for Q1 using
the year-agreement evidence. Do not claim that the whole dataset is clean or that proposed
corrections have been completed.

**Suggested reasoning:** A Passed date-order check establishes the ordering of its checked
pairs. Untested records still lack a usable pair. Year disagreements require reviewing source
definitions and selecting an appropriate year field before making annual comparisons.

**Optional next-step note:** A severity judgement, if your assessment asks for it, should explain
the effect on the selected question. The guide deliberately avoids automatic percentage-based priorities.


## Block 7 · Review the complete inspection handover

**Objective:** Finish with a study resource and a report that another reader can follow.

Before moving to cleaning, explain these points in your own words:

1. Where each rule came from and why it matters to an analysis question.
2. What Passed, Flagged and Untested mean for one record and for a check summary.
3. Why one issue row can have many evidence rows.
4. Why the same record can appear in more than one check's evidence.
5. Which evidence files are complete and which workbook sheets are previews.
6. Which proposed actions still need investigation before treatment.

**Handover:** Keep the unchanged source, these notebooks, `Check_Catalogue.csv`,
`inspection_helpers.py`, the reference lists and the `Outputs` folder together. The code
can be rerun from the source. Saved outputs make dependencies explicit across kernels.
A changed source requires regenerating earlier results before producing the report.

**Learning boundary:** These selected checks support an introductory inspection. Other
datasets and analysis questions may need different checks. You should be able to explain
each check here, its scope, its result and the action that the evidence supports.
